In [1]:
try:
    import google.colab
    inColab = True
except ImportError:
    inColab = False

if inColab == True:
    # !pip install -U accelerate==0.32.1 peft==0.12.0 bitsandbytes==0.43.3 transformers==4.44.2 trl==0.9.6
    !pip install -U pandas==2.2.2 numpy==2.0.2 scipy==1.14.1 accelerate==1.6.0 peft==0.15.2 bitsandbytes==0.45.5 transformers==4.51.3 trl==0.16.1

if inColab == True:
    !pip install -U nest-asyncio==1.6.0 pyngrok==7.2.4 uvicorn==0.34.2 fastapi==0.115.12

### 라이브러리 선언하기

In [2]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging
)
from transformers import AutoConfig,AutoModel
import torch
from peft import PeftModel, PeftConfig
import uvicorn
import huggingface_hub

In [3]:
# huggingface_hub.login()

### 1. 모델 불러오기

In [4]:
"""### ★★★ 수정 포인트 ★★★"""

## base 모델
# base_model = "hyokwan/familidata_llama31"
# base_model = "hyokwan/llama31_famili_2025"
base_model = "hyokwan/pharos_2b"
### 베이스모델 불러오기
baseModel = AutoModelForCausalLM.from_pretrained(
    base_model,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map= "auto" # T4 GPU 사용 시
    # device_map= {"": 0} # L4 이상 GRU 사용시
)

### 토크나이저 불러오기
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

### 추론 함수

In [5]:
"""### 어뎁터 연결 모델 정상작동 확인"""

DEFAULT_SYSTEM_MESSAGE = "당신은 문제를 정확하게 답변하는 AI입니다."
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# 시스템 메시지 + 문제(질문) 기반 답변 생성 함수

def generate_gemma_answer(model, tokenizer, user_message, system_message=DEFAULT_SYSTEM_MESSAGE, device=DEVICE,
                          max_new_tokens=512, temperature=0.2, top_p=0.95, top_k=50):
    if system_message:
        prompt = (
            f"<start_of_turn>system\n{system_message}<end_of_turn>\n"
            f"<start_of_turn>user\n{user_message}<end_of_turn>\n"
            f"<start_of_turn>model\n"
        )
    else:
        prompt = (
            f"<start_of_turn>user\n{user_message}<end_of_turn>\n"
            f"<start_of_turn>model\n"
        )
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    input_len = inputs["input_ids"].shape[1]
    model.to(device)
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    # 전체 디코딩
    # return tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 답변 토큰만 디코딩
    answer_tokens = outputs[0][input_len:]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

In [6]:
# 문제(질문) 예시
user_message = (
    "hkcode 유튜브 채널은 누가 운영하나요?<end_of_turn>\n"
)
# 시스템 메시지(옵션)
system_message = DEFAULT_SYSTEM_MESSAGE
# 답변 생성
output = generate_gemma_answer(baseModel, tokenizer, user_message, system_message)
print("\n[질문]\n", user_message)
print("\n[답변]\n", output)


[질문]
 hkcode 유튜브 채널은 누가 운영하나요?<end_of_turn>


[답변]
 한국폴리텍대학 스마트금융과 김효관 교수가 운영합니다.


In [7]:
print("\n[질문]\n", user_message)
print("\n[답변]\n", output)


[질문]
 hkcode 유튜브 채널은 누가 운영하나요?<end_of_turn>


[답변]
 한국폴리텍대학 스마트금융과 김효관 교수가 운영합니다.


In [8]:
# """### 정상동작 확인 #2"""

# 서버 관리용 fastapi 의존 라이브러리
import uvicorn

# fast api 라이브러리
from fastapi import FastAPI

# 데이터프레임 및 수 처리 라이브러리
import pandas as pd
import numpy as np
# 인터페이스 데이터 관리를 위한 라이브러리
from pydantic import BaseModel

from fastapi.middleware.cors import CORSMiddleware

### CORS 설정

In [9]:
origins = ["*"]

app = FastAPI(title="ML API")

# CORS 미들웨어 추가
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # 모든 origin 허용
    allow_credentials=True,
    allow_methods=["GET", "POST", "PUT", "DELETE"],
    allow_headers=["*"],
)

### 인터페이스 정의

In [23]:
class InDataset(BaseModel):
    question : str

### 엔드포인트 #1

In [ ]:
# STEP1: Indatast 클래스 타입으로 데이터를 받는다

In [31]:
@app.post("/predict", status_code=200)
async def predict_tf(x: InDataset):

    x = InDataset(question="스마트금융과 어디에 위치하나요?")
    question = x
    # STEP2: 받은 데이터를 generate_gemma_answer 함수에 대입한다)
    response = generate_gemma_answer(baseModel, tokenizer, question, system_message)
    result = {"prediction": response}
    # STEP3: 함수결과를 RETURN 한다. {"prediction": 예측결과}
    return result

In [32]:
@app.get('/')
async def root():
    return {"message": "online"}

In [ ]:
import nest_asyncio
nest_asyncio.apply()

# Uvicorn 실행
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=9999, log_level="debug")

INFO:     Started server process [25960]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9999 (Press CTRL+C to quit)


INFO:     127.0.0.1:55919 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:55919 - "GET /favicon.ico HTTP/1.1" 404 Not Found


INFO:     127.0.0.1:55646 - "POST /predict HTTP/1.1" 200 OK
